In [1]:
!pip -q install sentence-transformers faiss-cpu rank-bm25 pyarrow pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import re
import json
import math
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from tqdm.auto import tqdm

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

In [4]:
BASE_DIR = "/content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output"

SAN_PHAM_PATH = os.path.join(BASE_DIR, "san_pham.parquet")

ARTIFACT_DIR = os.path.join(BASE_DIR, "semantic_search_artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("SAN_PHAM_PATH =", SAN_PHAM_PATH)
print("ARTIFACT_DIR   =", ARTIFACT_DIR)
print("Ton tai file san_pham.parquet:", os.path.exists(SAN_PHAM_PATH))

SAN_PHAM_PATH = /content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/san_pham.parquet
ARTIFACT_DIR   = /content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/semantic_search_artifacts
Ton tai file san_pham.parquet: True


In [5]:
dataset = ds.dataset(SAN_PHAM_PATH, format="parquet")
schema = dataset.schema

print("=== CAC COT TRONG san_pham.parquet ===")
for field in schema:
    print("-", field.name, ":", field.type)

=== CAC COT TRONG san_pham.parquet ===
- asin : string
- average_rating : double
- categories : string
- description : string
- details : string
- features : string
- images : string
- main_category : string
- parent_asin : string
- price : string
- rating_number : int64
- store : string
- timestamp : string
- title : string
- top_reviews : string
- url : string
- author : string
- bought_together : string
- subtitle : string
- videos : string


In [6]:
def pick_existing_column(all_columns, candidates):
    for c in candidates:
        if c in all_columns:
            return c
    return None

all_columns = [field.name for field in schema]

TITLE_COL = pick_existing_column(all_columns, ["title", "product_title", "name"])
BRAND_COL = pick_existing_column(all_columns, ["brand"])
CATEGORY_COL = pick_existing_column(all_columns, ["category", "main_category", "categories"])
DETAILS_COL = pick_existing_column(all_columns, ["details", "description", "features"])
PRICE_COL = pick_existing_column(all_columns, ["price"])
ASIN_COL = pick_existing_column(all_columns, ["asin", "parent_asin"])

print("TITLE_COL   =", TITLE_COL)
print("BRAND_COL   =", BRAND_COL)
print("CATEGORY_COL=", CATEGORY_COL)
print("DETAILS_COL =", DETAILS_COL)
print("PRICE_COL   =", PRICE_COL)
print("ASIN_COL    =", ASIN_COL)

TITLE_COL   = title
BRAND_COL   = None
CATEGORY_COL= main_category
DETAILS_COL = details
PRICE_COL   = price
ASIN_COL    = asin


In [7]:
N_SAMPLE = 10000

selected_cols = [c for c in [ASIN_COL, TITLE_COL, BRAND_COL, CATEGORY_COL, DETAILS_COL, PRICE_COL] if c is not None]

table = dataset.to_table(columns=selected_cols)
df = table.to_pandas()

print("Shape truoc loc:", df.shape)
df.head(3)

Shape truoc loc: (2410915, 5)


,asin,title,main_category,details,price
0,0528881469,Rand McNally 528881469 7-inch Intelliroute TND...,All Electronics,"{""Are Batteries Included"":""Yes"",""Brand"":""Rand ...",None
1,0594515939,nook Charging Dock - Barnes and Nobles,All Electronics,"{""Best Sellers Rank"":{""eBook Reader Power Adap...",None
2,0783428006,Geochron World Watch,Computers,"{""Brand"":""Geochron"",""Date First Available"":""Ju...",None


In [8]:
df = df.copy()

# Bo dong khong co title
if TITLE_COL is not None:
    df = df[df[TITLE_COL].notna()]

# Drop duplicate theo asin neu co
if ASIN_COL is not None:
    df = df.drop_duplicates(subset=[ASIN_COL])
else:
    df = df.drop_duplicates()

df = df.reset_index(drop=True)

print("Shape sau loc:", df.shape)
df.head(3)

Shape sau loc: (2410915, 5)


,asin,title,main_category,details,price
0,0528881469,Rand McNally 528881469 7-inch Intelliroute TND...,All Electronics,"{""Are Batteries Included"":""Yes"",""Brand"":""Rand ...",None
1,0594515939,nook Charging Dock - Barnes and Nobles,All Electronics,"{""Best Sellers Rank"":{""eBook Reader Power Adap...",None
2,0783428006,Geochron World Watch,Computers,"{""Brand"":""Geochron"",""Date First Available"":""Ju...",None


In [9]:
def clean_text(x):
    if pd.isna(x):
        return ""
    if isinstance(x, list):
        x = " ".join([str(i) for i in x if pd.notna(i)])
    elif isinstance(x, dict):
        x = " ".join([f"{k}: {v}" for k, v in x.items()])
    else:
        x = str(x)

    x = x.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_price(x):
    if pd.isna(x):
        return ""
    x = str(x)
    m = re.search(r"(\d+(\.\d+)?)", x.replace(",", ""))
    return m.group(1) if m else ""

In [10]:
def build_retrieval_text(row):
    title = clean_text(row[TITLE_COL]) if TITLE_COL else ""
    brand = clean_text(row[BRAND_COL]) if BRAND_COL else ""
    category = clean_text(row[CATEGORY_COL]) if CATEGORY_COL else ""
    details = clean_text(row[DETAILS_COL]) if DETAILS_COL else ""
    price = normalize_price(row[PRICE_COL]) if PRICE_COL else ""

    parts = [
        title,
        f"brand {brand}" if brand else "",
        f"category {category}" if category else "",
        f"details {details}" if details else "",
        f"price {price}" if price else "",
    ]

    text = " | ".join([p for p in parts if p])
    return text.strip()

df["retrieval_text"] = df.apply(build_retrieval_text, axis=1)

# Bo cac dong text qua ngan
df["retrieval_text_len"] = df["retrieval_text"].str.len()
df = df[df["retrieval_text_len"] > 20].reset_index(drop=True)

print("Shape sau khi tao retrieval_text:", df.shape)
df[[TITLE_COL, "retrieval_text"]].head(5)

Shape sau khi tao retrieval_text: (2410915, 7)


,title,retrieval_text
0,Rand McNally 528881469 7-inch Intelliroute TND...,Rand McNally 528881469 7-inch Intelliroute TND...
1,nook Charging Dock - Barnes and Nobles,nook Charging Dock - Barnes and Nobles | categ...
2,Geochron World Watch,Geochron World Watch | category Computers | de...
3,LG Volt (OTG) Micro-USB to USB 2.0 Adapter Hig...,LG Volt (OTG) Micro-USB to USB 2.0 Adapter Hig...
4,Essential 64GB Sony Xperia miro Micro SDHC Car...,Essential 64GB Sony Xperia miro Micro SDHC Car...


In [11]:
if len(df) > N_SAMPLE:
    df_demo = df.sample(N_SAMPLE, random_state=42).reset_index(drop=True)
else:
    df_demo = df.copy().reset_index(drop=True)

print("So san pham demo:", len(df_demo))
df_demo.head(3)

So san pham demo: 10000


,asin,title,main_category,details,price,retrieval_text,retrieval_text_len
0,B075Z3V4G9,ALXCD Ear Adapters for IE 800 Ear Canal Headph...,Home Audio & Theater,"{""Color"":""Black"",""Date First Available"":""Septe...",None,ALXCD Ear Adapters for IE 800 Ear Canal Headph...,480
1,B082YH9WMC,Spigen ProFlex EZ FIT Screen Protector Designe...,Cell Phones & Accessories,"{""Best Sellers Rank"":{""Cell Phones & Accessori...",None,Spigen ProFlex EZ FIT Screen Protector Designe...,726
2,B07NRZ8H5W,"BEBEST-Galaxy S10e Case, Galaxy S10e Clear Cas...",Cell Phones & Accessories,"{""Brand"":""BEBEST"",""Compatible Phone Models"":""S...",None,"BEBEST-Galaxy S10e Case, Galaxy S10e Clear Cas...",686


In [12]:
DEMO_PARQUET_PATH = os.path.join(ARTIFACT_DIR, "demo_products.parquet")
df_demo.to_parquet(DEMO_PARQUET_PATH, index=False)

print("Da luu:", DEMO_PARQUET_PATH)

Da luu: /content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/semantic_search_artifacts/demo_products.parquet


In [13]:
def simple_tokenize(text):
    text = clean_text(text).lower()
    text = re.sub(r"[^0-9a-zA-ZÀ-ỹ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

corpus_tokens = [simple_tokenize(x) for x in df_demo["retrieval_text"].tolist()]
bm25 = BM25Okapi(corpus_tokens)

print("Da tao BM25 index cho", len(corpus_tokens), "san pham")

Da tao BM25 index cho 10000 san pham


In [14]:
def search_bm25(query, top_k=5):
    q_tokens = simple_tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = df_demo.iloc[top_idx].copy()
    results["bm25_score"] = scores[top_idx]
    return results

In [15]:
MODEL_1_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
model_1 = SentenceTransformer(MODEL_1_NAME)

print("Loaded model 1:", MODEL_1_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded model 1: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


In [16]:
texts = df_demo["retrieval_text"].tolist()

emb_1 = model_1.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Shape emb_1:", emb_1.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Shape emb_1: (10000, 768)


In [17]:
dim_1 = emb_1.shape[1]
index_1 = faiss.IndexFlatIP(dim_1)   # Inner Product, do da normalize roi
index_1.add(emb_1.astype("float32"))

print("FAISS index 1 size:", index_1.ntotal)

FAISS index 1 size: 10000


In [18]:
def search_dense_model_1(query, top_k=5):
    q_emb = model_1.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index_1.search(q_emb, top_k)
    idx = indices[0]
    sc = scores[0]

    results = df_demo.iloc[idx].copy()
    results["dense_score_model_1"] = sc
    return results

In [19]:
MODEL_2_NAME = "sentence-transformers/LaBSE"
model_2 = SentenceTransformer(MODEL_2_NAME)

print("Loaded model 2:", MODEL_2_NAME)

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Loaded model 2: sentence-transformers/LaBSE


In [20]:
emb_2 = model_2.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Shape emb_2:", emb_2.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Shape emb_2: (10000, 768)


In [21]:
dim_2 = emb_2.shape[1]
index_2 = faiss.IndexFlatIP(dim_2)
index_2.add(emb_2.astype("float32"))

print("FAISS index 2 size:", index_2.ntotal)

FAISS index 2 size: 10000


In [22]:
def search_dense_model_2(query, top_k=5):
    q_emb = model_2.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index_2.search(q_emb, top_k)
    idx = indices[0]
    sc = scores[0]

    results = df_demo.iloc[idx].copy()
    results["dense_score_model_2"] = sc
    return results

In [23]:
def minmax_scale(arr):
    arr = np.array(arr, dtype=float)
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-9:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

def search_hybrid(query, top_k=5, alpha=0.5, dense_top_n=100, bm25_top_n=100):
    # BM25 scores
    q_tokens = simple_tokenize(query)
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_idx = np.argsort(bm25_scores)[::-1][:bm25_top_n]

    # Dense scores model 1
    q_emb = model_1.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    dense_scores, dense_indices = index_1.search(q_emb, dense_top_n)
    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    # Union candidate
    candidate_idx = list(set(bm25_idx.tolist()) | set(dense_indices.tolist()))

    # Map dense score
    dense_map = {int(i): float(s) for i, s in zip(dense_indices, dense_scores)}
    bm25_map = {int(i): float(bm25_scores[i]) for i in candidate_idx}

    bm25_arr = np.array([bm25_map[i] for i in candidate_idx], dtype=float)
    dense_arr = np.array([dense_map.get(i, 0.0) for i in candidate_idx], dtype=float)

    bm25_scaled = minmax_scale(bm25_arr)
    dense_scaled = minmax_scale(dense_arr)

    final_scores = alpha * dense_scaled + (1 - alpha) * bm25_scaled

    order = np.argsort(final_scores)[::-1][:top_k]
    final_idx = [candidate_idx[i] for i in order]

    results = df_demo.iloc[final_idx].copy()
    results["hybrid_score"] = final_scores[order]
    return results

In [24]:
def show_results(results, title_col=TITLE_COL, price_col=PRICE_COL, max_rows=5):
    display_cols = []
    for c in [ASIN_COL, title_col, BRAND_COL, CATEGORY_COL, price_col, "retrieval_text"]:
        if c is not None and c in results.columns:
            display_cols.append(c)

    score_cols = [c for c in results.columns if "score" in c.lower()]
    display_cols += score_cols

    return results[display_cols].head(max_rows)

In [25]:
test_query = "laptop mỏng nhẹ cho sinh viên"

print("=== BM25 ===")
display(show_results(search_bm25(test_query, top_k=5)))

print("=== Dense Model 1 ===")
display(show_results(search_dense_model_1(test_query, top_k=5)))

print("=== Dense Model 2 ===")
display(show_results(search_dense_model_2(test_query, top_k=5)))

print("=== Hybrid ===")
display(show_results(search_hybrid(test_query, top_k=5, alpha=0.6)))

=== BM25 ===


,asin,title,main_category,price,retrieval_text,bm25_score
6120,B08THFDGWN,"ALZHIJ Laptop Stand Adjustable,Ergonomic Alumi...",Computers,None,"ALZHIJ Laptop Stand Adjustable,Ergonomic Alumi...",5.102376
9454,B09BJJ8DVY,ENUSUNG Vertical Laptop Stand Adjustable Doubl...,Computers,16.99,ENUSUNG Vertical Laptop Stand Adjustable Doubl...,5.075721
202,B0BSR6GVPF,Vooray 15-Inch Laptop Sleeve – Computer Sleeve...,Computers,13.0,Vooray 15-Inch Laptop Sleeve – Computer Sleeve...,4.769802
2143,B08TMCS5ST,"evron Laptop Stand for Desk,Width Adjustable L...",Computers,None,"evron Laptop Stand for Desk,Width Adjustable L...",4.723426
2603,B00FW6CUT6,"LP140WF1(SP)(B1) Laptop LCD Screen 14.0"" Full-...",Computers,52.87,"LP140WF1(SP)(B1) Laptop LCD Screen 14.0"" Full-...",4.704413


=== Dense Model 1 ===


,asin,title,main_category,price,retrieval_text,dense_score_model_1
6120,B08THFDGWN,"ALZHIJ Laptop Stand Adjustable,Ergonomic Alumi...",Computers,None,"ALZHIJ Laptop Stand Adjustable,Ergonomic Alumi...",0.660613
8812,B08S37YLHZ,Foldable&Portable Aluminum Laptop Stand for De...,All Electronics,None,Foldable&Portable Aluminum Laptop Stand for De...,0.660120
9454,B09BJJ8DVY,ENUSUNG Vertical Laptop Stand Adjustable Doubl...,Computers,16.99,ENUSUNG Vertical Laptop Stand Adjustable Doubl...,0.658723
202,B0BSR6GVPF,Vooray 15-Inch Laptop Sleeve – Computer Sleeve...,Computers,13.0,Vooray 15-Inch Laptop Sleeve – Computer Sleeve...,0.656905
5404,B08GCD3BJM,Clearance：Portable Laptop Stand 10 Angles Adju...,Computers,None,Clearance：Portable Laptop Stand 10 Angles Adju...,0.654265


=== Dense Model 2 ===


,asin,title,main_category,price,retrieval_text,dense_score_model_2
6799,B09P54KSBF,Laptop Backpack starry Tablet Bag Travel Busin...,Computers,35.99,Laptop Backpack starry Tablet Bag Travel Busin...,0.414960
6205,B01MXPYBM0,Canvaslife Light Grey Lightweight Laptop Brief...,Computers,None,Canvaslife Light Grey Lightweight Laptop Brief...,0.412128
8130,B08HQ7TQRC,"2021 Newest HP Chromebook 14"" Full-HD IPS Lapt...",Computers,None,"2021 Newest HP Chromebook 14"" Full-HD IPS Lapt...",0.410167
5703,B09WRFP53S,YAMTION Backpack for School College Work Busin...,Computers,None,YAMTION Backpack for School College Work Busin...,0.406535
6817,B07ZNVJKRJ,"Laptop Backpack for Women, Lekesky Work Bags B...",All Electronics,None,"Laptop Backpack for Women, Lekesky Work Bags B...",0.395639


=== Hybrid ===


,asin,title,main_category,price,retrieval_text,hybrid_score
6120,B08THFDGWN,"ALZHIJ Laptop Stand Adjustable,Ergonomic Alumi...",Computers,None,"ALZHIJ Laptop Stand Adjustable,Ergonomic Alumi...",1.000000
9454,B09BJJ8DVY,ENUSUNG Vertical Laptop Stand Adjustable Doubl...,Computers,16.99,ENUSUNG Vertical Laptop Stand Adjustable Doubl...,0.996194
202,B0BSR6GVPF,Vooray 15-Inch Laptop Sleeve – Computer Sleeve...,Computers,13.0,Vooray 15-Inch Laptop Sleeve – Computer Sleeve...,0.970560
2143,B08TMCS5ST,"evron Laptop Stand for Desk,Width Adjustable L...",Computers,None,"evron Laptop Stand for Desk,Width Adjustable L...",0.952004
6832,B0095B3WGE,Laptop Skin Shop 15.6 inch Laptop Sleeve Bag C...,Computers,None,Laptop Skin Shop 15.6 inch Laptop Sleeve Bag C...,0.938601


In [26]:
sample_queries = [
    "laptop học lập trình",
    "điện thoại pin trâu",
    "tai nghe chống ồn",
    "chuột không dây văn phòng",
    "màn hình chơi game",
    "loa bluetooth nhỏ gọn",
    "bàn phím cơ giá rẻ"
]

for q in sample_queries:
    print("\n" + "="*80)
    print("QUERY:", q)

    print("\n[BM25]")
    display(show_results(search_bm25(q, top_k=3), max_rows=3))

    print("\n[DENSE MODEL 1]")
    display(show_results(search_dense_model_1(q, top_k=3), max_rows=3))

    print("\n[DENSE MODEL 2]")
    display(show_results(search_dense_model_2(q, top_k=3), max_rows=3))

    print("\n[HYBRID]")
    display(show_results(search_hybrid(q, top_k=3, alpha=0.6), max_rows=3))


QUERY: laptop học lập trình

[BM25]


,asin,title,main_category,price,retrieval_text,bm25_score
6120,B08THFDGWN,"ALZHIJ Laptop Stand Adjustable,Ergonomic Alumi...",Computers,None,"ALZHIJ Laptop Stand Adjustable,Ergonomic Alumi...",5.102376
9454,B09BJJ8DVY,ENUSUNG Vertical Laptop Stand Adjustable Doubl...,Computers,16.99,ENUSUNG Vertical Laptop Stand Adjustable Doubl...,5.075721
202,B0BSR6GVPF,Vooray 15-Inch Laptop Sleeve – Computer Sleeve...,Computers,13.0,Vooray 15-Inch Laptop Sleeve – Computer Sleeve...,4.769802



[DENSE MODEL 1]


,asin,title,main_category,price,retrieval_text,dense_score_model_1
6206,B09WF5G3MT,"HP 2022 Newest Notebook Laptop, 17.3'' HD+ Tou...",Computers,639.0,"HP 2022 Newest Notebook Laptop, 17.3'' HD+ Tou...",0.552509
3492,B00QQM2EJQ,CyberpowerPC Fangbook III HFX6-500 15.6-Inch L...,Computers,None,CyberpowerPC Fangbook III HFX6-500 15.6-Inch L...,0.542267
1195,B09L6V633P,"Lenovo ThinkBook 13s Gen 2 Laptop, WQXGA (2560...",Computers,None,"Lenovo ThinkBook 13s Gen 2 Laptop, WQXGA (2560...",0.532974



[DENSE MODEL 2]


...

,asin,title,main_category,price,retrieval_text,dense_score_model_1
7164,B071X5YYL3,wangpeng® New Black US Laptop Keyboard for HP ...,Computers,25.0,wangpeng® New Black US Laptop Keyboard for HP ...,0.652204
6897,B01MZHVELM,"Mechanical Keyboard with Blue Switches,LINGBAO...",Computers,None,"Mechanical Keyboard with Blue Switches,LINGBAO...",0.645826
5617,B08FTFXWY1,SOOTOP Steampunk Typewriter Keycaps Pudding Ke...,Computers,15.59,SOOTOP Steampunk Typewriter Keycaps Pudding Ke...,0.634984



[DENSE MODEL 2]


,asin,title,main_category,price,retrieval_text,dense_score_model_2
7164,B071X5YYL3,wangpeng® New Black US Laptop Keyboard for HP ...,Computers,25.0,wangpeng® New Black US Laptop Keyboard for HP ...,0.403658
5395,B07ZX8R8RC,"Magnetic Folding Bluetooth Keyboard, Rechargea...",Computers,22.51,"Magnetic Folding Bluetooth Keyboard, Rechargea...",0.372677
9333,B0BLHQC95V,Logitech MX Mechanical Full-Size Illuminated W...,Computers,268.99,Logitech MX Mechanical Full-Size Illuminated W...,0.354402



[HYBRID]


,asin,title,main_category,price,retrieval_text,hybrid_score
7164,B071X5YYL3,wangpeng® New Black US Laptop Keyboard for HP ...,Computers,25.0,wangpeng® New Black US Laptop Keyboard for HP ...,0.600000
6897,B01MZHVELM,"Mechanical Keyboard with Blue Switches,LINGBAO...",Computers,None,"Mechanical Keyboard with Blue Switches,LINGBAO...",0.594132
5617,B08FTFXWY1,SOOTOP Steampunk Typewriter Keycaps Pudding Ke...,Computers,15.59,SOOTOP Steampunk Typewriter Keycaps Pudding Ke...,0.584158


In [27]:
eval_queries = pd.DataFrame({
    "query_id": list(range(1, 21)),
    "query": [
        "laptop học lập trình",
        "laptop văn phòng pin lâu",
        "điện thoại chụp ảnh đẹp",
        "điện thoại pin trâu",
        "tai nghe chống ồn",
        "tai nghe bluetooth giá rẻ",
        "chuột gaming nhẹ",
        "chuột không dây văn phòng",
        "bàn phím cơ giá rẻ",
        "màn hình chơi game 27 inch",
        "màn hình 2k thiết kế",
        "loa bluetooth di động",
        "sạc nhanh điện thoại",
        "ổ cứng di động dung lượng lớn",
        "camera an ninh trong nhà",
        "tablet học online",
        "đồng hồ thông minh theo dõi sức khỏe",
        "router wifi mạnh",
        "micro thu âm",
        "webcam họp online"
    ]
})

EVAL_QUERY_PATH = os.path.join(ARTIFACT_DIR, "eval_queries.csv")
eval_queries.to_csv(EVAL_QUERY_PATH, index=False)

print("Da luu:", EVAL_QUERY_PATH)
eval_queries.head()

Da luu: /content/drive/MyDrive/MiningMassiveData/crawl_amazon_metadata/metadata/output/semantic_search_artifacts/eval_queries.csv


,query_id,query
0,1,laptop học lập trình
1,2,laptop văn phòng pin lâu
2,3,điện thoại chụp ảnh đẹp
3,4,điện thoại pin trâu
4,5,tai nghe chống ồn


In [28]:
np.save(os.path.join(ARTIFACT_DIR, "embeddings_model_1.npy"), emb_1.astype("float32"))
np.save(os.path.join(ARTIFACT_DIR, "embeddings_model_2.npy"), emb_2.astype("float32"))

faiss.write_index(index_1, os.path.join(ARTIFACT_DIR, "faiss_model_1.index"))
faiss.write_index(index_2, os.path.join(ARTIFACT_DIR, "faiss_model_2.index"))

with open(os.path.join(ARTIFACT_DIR, "meta.json"), "w", encoding="utf-8") as f:
    json.dump({
        "demo_parquet": DEMO_PARQUET_PATH,
        "model_1_name": MODEL_1_NAME,
        "model_2_name": MODEL_2_NAME,
        "n_products": len(df_demo),
        "title_col": TITLE_COL,
        "brand_col": BRAND_COL,
        "category_col": CATEGORY_COL,
        "details_col": DETAILS_COL,
        "price_col": PRICE_COL,
        "asin_col": ASIN_COL
    }, f, ensure_ascii=False, indent=2)

print("Da luu xong artifact.")

Da luu xong artifact.
